In [24]:
from pyspark.sql import SparkSession

# 1. Crear la Sesión de Spark
# 'local[*]' significa que usará todos los núcleos de tu procesador para trabajar en paralelo
spark = SparkSession.builder \
    .appName("ImportarNombresKaggle") \
    .master("local[*]") \
    .getOrCreate()


In [25]:
# 2. Definir la ruta donde guardaste el CSV en tu máquina

forename_csv = r"C:\Users\Hugo\OneDrive\Documentos\MEGA\PROYECTOS\Personal proyects\nn_find_gender_by_name\data_source\archive (1)\forenames.csv"
ctry_code_csv = r"C:\Users\Hugo\OneDrive\Documentos\MEGA\PROYECTOS\Personal proyects\nn_find_gender_by_name\data_source\archive (1)\country_codes.csv"

In [26]:
print("Iniciando la lectura del CSV con Spark...")

# 3. Leer el CSV
# - header=True: Usa la primera fila como nombres de columna.
# - inferSchema=True: Spark analiza el archivo para detectar si los datos son texto, números, etc.
df_fn = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv(forename_csv)

Iniciando la lectura del CSV con Spark...


In [27]:
df_cc = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv(ctry_code_csv)

In [28]:
# 4. Mostrar el esquema (la estructura de las columnas)
print("\n--- Estructura del Dataset ---")
df_cc.printSchema()


--- Estructura del Dataset ---
root
 |-- country_code: string (nullable = true)
 |-- country_name: string (nullable = true)



In [29]:
# 4. Mostrar el esquema (la estructura de las columnas)
print("\n--- Estructura del Dataset ---")
df_fn.printSchema()


--- Estructura del Dataset ---
root
 |-- forename: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- country: string (nullable = true)
 |-- count: integer (nullable = true)



In [30]:
# 5. Mostrar las primeras 10 filas (¡Esto es una Acción y activa el procesamiento!)
print("\n--- Primeros 10 registros ---")
df_fn.show(10)


--- Primeros 10 registros ---
+--------+------+-------+------+
|forename|gender|country| count|
+--------+------+-------+------+
|      Md|     M|     AE|132181|
|Muhammad|     M|     AE| 84884|
|Mohammed|     M|     AE| 51575|
|   Abdul|     M|     AE| 48408|
|     Ali|     M|     AE| 40866|
|Mohammad|     M|     AE| 39532|
|   Ahmed|     M|     AE| 31549|
| Mohamed|     M|     AE| 30575|
|    محمد|     M|     AE| 25879|
|   Ahmad|     M|     AE| 15871|
+--------+------+-------+------+
only showing top 10 rows


In [33]:
print("\n--- Primeros 10 registros ---")
df_cc.show(10)


--- Primeros 10 registros ---
+------------+--------------------+
|country_code|        country_name|
+------------+--------------------+
|          AE|United Arab Emirates|
|          AF|         Afghanistan|
|          AL|             Albania|
|          AO|              Angola|
|          AR|           Argentina|
|          AT|             Austria|
|          AZ|          Azerbaijan|
|          BD|          Bangladesh|
|          BE|             Belgium|
|          BF|        Burkina Faso|
+------------+--------------------+
only showing top 10 rows


In [31]:
# 6. Contar el total de registros
total_filas = df_fn.count()
print(f"\nEl dataset tiene un total de: {total_filas:,} filas.")


El dataset tiene un total de: 12,470,920 filas.


In [32]:
df_fn.describe().show()

+-------+----------+--------+--------+------------------+
|summary|  forename|  gender| country|             count|
+-------+----------+--------+--------+------------------+
|  count|  12470629|10051785|12443348|          12470920|
|   mean|       NaN|    NULL|    NULL|36.390102895375804|
| stddev|       NaN|    NULL|    NULL|1310.8466078337094|
|    min|      صلاح|       F|      AE|                 2|
|    max|Ｚａｈｒａ|       M|      ZA|           1584172|
+-------+----------+--------+--------+------------------+



In [37]:
df_merge = df_fn.join(df_cc, df_fn['country'] == df_cc['country_code'], 'inner')
df_merge.show(3)

+--------+------+-------+------+------------+--------------------+
|forename|gender|country| count|country_code|        country_name|
+--------+------+-------+------+------------+--------------------+
|      Md|     M|     AE|132181|          AE|United Arab Emirates|
|Muhammad|     M|     AE| 84884|          AE|United Arab Emirates|
|Mohammed|     M|     AE| 51575|          AE|United Arab Emirates|
+--------+------+-------+------+------------+--------------------+
only showing top 3 rows


In [42]:
from pyspark.sql.functions import col

paises_latam = ['CO', 'MX', 'AR', 'VE', 'PE', 'CL']
df_latam = df_merge.filter(col('country').isin(paises_latam))
df_latam.show(3)

+--------+------+-------+-----+------------+------------+
|forename|gender|country|count|country_code|country_name|
+--------+------+-------+-----+------------+------------+
|    Juan|     M|     AR|21823|          AR|   Argentina|
|   Maria|     F|     AR|20758|          AR|   Argentina|
|  Carlos|     M|     AR|19728|          AR|   Argentina|
+--------+------+-------+-----+------------+------------+
only showing top 3 rows
